In [0]:

#  Locate JSON redemption files in ADLS Gen2

json_path = "abfss://raw@incubytedatalake01.dfs.core.windows.net/redemption"

display(dbutils.fs.ls(json_path))

[]

In [0]:
# Check files inside redemption folder

json_path = "abfss://raw@incubytedatalake01.dfs.core.windows.net/redemption/"

files = dbutils.fs.ls(json_path)

for file in files:
    print("File:", file.name)
    print("Size:", file.size, "bytes")

File: skypoints_redemption_sample.json
Size: 391 bytes


In [0]:
#  Verify JSON file in ADLS Gen2

json_path = "abfss://raw@incubytedatalake01.dfs.core.windows.net/redemption/"

display(dbutils.fs.ls(json_path))

path,name,size,modificationTime
abfss://raw@incubytedatalake01.dfs.core.windows.net/redemption/skypoints_redemption_sample.json,skypoints_redemption_sample.json,391,1795252213000


In [0]:
#  Read JSON redemption data

redemption_df = (
    spark.read
    .option("multiline", "true")
    .json(json_path)
)

display(redemption_df)

feed_date,member_id,redemptions
20240115,223457,"List(List(12000, AeroLink, COMPLETED, 20240110, RX10091), List(5000, SkyPoints, PENDING, 20240113, RX10092))"


In [0]:
# Check whether the JSON file was read successfully
display(redemption_df)

feed_date,member_id,redemptions
20240115,223457,"List(List(12000, AeroLink, COMPLETED, 20240110, RX10091), List(5000, SkyPoints, PENDING, 20240113, RX10092))"


In [0]:
redemption_df.createOrReplaceTempView("redemption_df")

print("Temporary view created successfully")

Temporary view created successfully


In [0]:
%sql
SELECT * FROM redemption_df;

feed_date,member_id,redemptions
20240115,223457,"List(List(12000, AeroLink, COMPLETED, 20240110, RX10091), List(5000, SkyPoints, PENDING, 20240113, RX10092))"


In [0]:
%sql
--Flattened sql query
CREATE OR REPLACE TEMP VIEW flattened_redemptions AS
SELECT
    member_id,
    feed_date,
    redemption.txn_id AS Transaction_ID,
    redemption.txn_date AS Transaction_Date,
    redemption.partner AS Partner,
    redemption.miles_redeemed AS Miles_Redeemed,
    redemption.status AS Transaction_Status
FROM redemption_df
LATERAL VIEW EXPLODE(redemptions) exploded_table AS redemption;

In [0]:
%sql SELECT * FROM flattened_redemptions;


member_id,feed_date,Transaction_ID,Transaction_Date,Partner,Miles_Redeemed,Transaction_Status
223457,20240115,RX10091,20240110,AeroLink,12000,COMPLETED
223457,20240115,RX10092,20240113,SkyPoints,5000,PENDING


In [0]:
%sql
--Save the flattened data as a Gold Delta table
CREATE OR REPLACE TABLE delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`
USING DELTA
AS
SELECT *
FROM flattened_redemptions;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`;

member_id,feed_date,Transaction_ID,Transaction_Date,Partner,Miles_Redeemed,Transaction_Status
223457,20240115,RX10091,20240110,AeroLink,12000,COMPLETED
223457,20240115,RX10092,20240113,SkyPoints,5000,PENDING


In [0]:
%sql
--Joining redemtion data with member data
CREATE OR REPLACE TEMP VIEW member_redemption_join AS

SELECT
    r.member_id,
    r.Transaction_ID,
    r.Transaction_Date,
    r.Partner,
    r.Miles_Redeemed,
    r.Transaction_Status,
    m.Member_Name,
    m.Source_Country

FROM flattened_redemptions r

LEFT JOIN delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/` m

ON r.member_id = CAST(m.Member_ID AS STRING);

In [0]:
%sql
SELECT * FROM member_redemption_join;

member_id,Transaction_ID,Transaction_Date,Partner,Miles_Redeemed,Transaction_Status,Member_Name,Source_Country
223457,RX10091,20240110,AeroLink,12000,COMPLETED,null,null
223457,RX10092,20240113,SkyPoints,5000,PENDING,null,null


In [0]:
%sql
--Check matched and unmatched records
SELECT
    COUNT(*) AS Total_Transactions,
    COUNT(Member_Name) AS Matched_Transactions,
    COUNT(*) - COUNT(Member_Name) AS Unmatched_Transactions
FROM member_redemption_join;


Total_Transactions,Matched_Transactions,Unmatched_Transactions
2,0,2
